# MSLN Spatial Heterogeneity for ADC Candidate Stratification

## What this notebook does

**Mesothelin (MSLN)** is the target of multiple antibody-drug conjugates (ADCs)
in pancreatic cancer — including Anetumab ravtansine, BMS-986148, and
RC88. Patient selection currently relies on MSLN IHC
to confirm target expression. While bulk RNA-seq MSLN levels correlate with
IHC positivity (BASECAMP-1, JCO 2025), neither approach captures **spatial
context**: bulk cannot distinguish high MSLN in 40% of cells from moderate
MSLN uniformly distributed, and IHC scoring collapses spatial heterogeneity
into a single percentage.

More critically, **expression alone is not sufficient** — if the ADC cannot
physically reach MSLN-expressing tumor cells (e.g., because they are encased
in dense desmoplastic stroma), the drug will not be effective regardless of
target abundance.

This notebook demonstrates how **H&E & bulk RNAseq-predicted spatial gene expression**
from the Bioptimus SDK reveals information that IHC and bulk RNA-seq cannot provide.

### The problem with current patient selection

Both patients in this demo are MSLN-positive by IHC criteria and in the
**top quartile of bulk MSLN expression** in TCGA-PAAD. Yet their **spatial
patterns** are dramatically different, both in terms of MSLN expression
profiles and the surrounding stromal environment:

- One has MSLN in accessible tumor regions (good ADC candidate).
- The other has MSLN trapped behind dense desmoplastic stroma (poor candidate).

### What we compute

1. **Spatial MSLN expression** — tile-level predicted gene expression from H&E.
2. **Desmoplastic score** — per-tile mean z-score of 5 stromal program genes†
   (ACTA2, COL1A1, POSTN, VCAN, LAMC2), each predicted well from H&E by M-Optimus.
3. **Stromal barrier metric** — fraction of MSLN-high tiles whose local
   neighborhood has high desmoplastic stroma → predicts ADC penetration
   difficulty.

†Stromal genes signature adapted from:
- Öhlund et al., 2017 (Journal of Experimental Medicine)
- Elyada et al., 2019 (Cancer Cell)
- Hirani et al., 2024 (Cancer Research Communications)

The panel is restricted to stromal-program genes that M-Optimus predicts well
from H&E; each is individually defensible as a marker of desmoplastic stroma.

---

## Prerequisites

To run this notebook you need a deployed **M-Optimus** inference server and the
Bioptimus SDK. See the [Quickstart](https://docs.bioptimus.com/documentation/quickstart)
to deploy a server (AWS SageMaker or on-premise container).

| Requirement | Details |
|-------------|--------|
| **Inference server** | An M-Optimus server reachable at `API_URL` (defaults to `http://localhost:8080`). Set the `API_URL` environment variable to point elsewhere — no need to edit the notebook. |
| **SDK** | `bioptimus-sdk` (installed in the first cell). |
| **Compute** | The SDK reads slides and dispatches tiles to the server; a GPU is required **server-side**, not in this notebook's kernel. |
| **Disk space** | ~4 GB for the 2 slides + ~2 MB for the TSV files + ~1 GB for outputs. |
| **Data** | 2 TCGA-PAAD slides + matched bulk RNA-seq, downloaded automatically from the public [GDC API](https://gdc.cancer.gov/). |
| **Runtime** | Prediction over both slides takes roughly 25–30 min per stage on a single-GPU server. |

### Documentation

| Resource | URL |
|---|---|
| M-Optimus model | https://docs.bioptimus.com/documentation/models/m-optimus |
| Bioptimus SDK | https://docs.bioptimus.com/guides/get-started/sdk |
| Inference pipeline | https://docs.bioptimus.com/guides/get-started/inference-pipeline |
| Visualizing results | https://docs.bioptimus.com/guides/get-started/visualizing-results |

### Data attribution & licensing

The diagnostic slides (`.svs`) and gene-quantification files (`.tsv`) used here
are **open-access TCGA-PAAD** data, downloaded at runtime from the public
[NCI GDC](https://gdc.cancer.gov/). Open-access TCGA data (histology images and
derived expression) carry **no restrictions on use in publications or
presentations**, and are governed by the
[NIH Genomic Data Sharing (GDS) Policy](https://osp.od.nih.gov/scientific-sharing/policies/):
do not attempt to re-identify participants, and acknowledge the source. Per the
[TCGA citation guidance](https://www.cancer.gov/ccg/research/genome-sequencing/tcga/using-tcga-data/citing):

> The results shown here are in whole or part based upon data generated by the
> TCGA Research Network: https://www.cancer.gov/tcga.

> **Intended use.** Bioptimus models are for **research use only** and are not
> approved medical devices. See
> [Responsible use](https://docs.bioptimus.com/documentation/resources/responsible-use).


## 0. Environment Setup

In [ ]:
import sys

assert sys.version_info >= (3, 12), f"Python 3.12+ is required, but found {sys.version}."
print(f"Python: {sys.version}")

# Pin the SDK for a reproducible demo (validated against 1.2.0).
%pip install "bioptimus-sdk==1.2.0"
%pip install matplotlib scipy


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from bioptimus.data.cohort import Cohort
from bioptimus.inference import Inference
from bioptimus.models.types import Models
from bioptimus.utils import (
    check_server,
    display_rna_summary,
    display_slide_info,
    download_if_needed,
    load_zarr_output,
    plot_gene_overlay,
    plot_gene_overlay_comparison,
    plot_gene_panel_overlay,
    plot_slide_and_mask,
    plot_top_gene_tiles,
    print_config,
    read_gene_mapping,
)


## 1. Download TCGA-PAAD Data

Downloads 2 diagnostic H&E slides (`.svs`) and matched bulk RNA-seq files
(`.tsv`) from the GDC API.

| Patient | ADC Candidate | SVS UUID | RNA-seq UUID |
|---------|--------------|----------|-------------|
| TCGA-F2-A8YN | Good | bb77b463-... | 35e2ac4f-... |
| TCGA-US-A77G | Poor | 93b92629-... | c5b35b34-... |

> **Note on RNA columns.** The GDC TSVs carry several count columns. This demo
> uses **`tpm_unstranded`** (TPM-normalized) throughout, which is the scale
> M-Optimus expects for bulk RNA. The `display_rna_summary` output below reports
> stats for the file's last numeric column (`fpkm_uq_unstranded`) purely as a
> quick file preview — it is *not* the column used for prediction.


In [ ]:
# All demo inputs and outputs live under .data/, the repo's gitignored scratch
# location for models and data (see .gitignore), so nothing large is committed.
DATA_ROOT = Path(".data") / "msln_adc_demo"
DEMO_DATA_DIR = DATA_ROOT / "demo_external_data"
DEMO_WSI_DIR = DEMO_DATA_DIR / "wsi"
DEMO_OMICS_DIR = DEMO_DATA_DIR / "omics"

DEMO_WSI_DIR.mkdir(parents=True, exist_ok=True)
DEMO_OMICS_DIR.mkdir(parents=True, exist_ok=True)

# TCGA-PAAD samples: 1 good ADC candidate + 1 poor ADC candidate.
GDC_BASE = "https://api.gdc.cancer.gov/data"

SAMPLES = {
    "TCGA-F2-A8YN": {
        "svs_uuid": "bb77b463-fe08-4513-a49d-aabd19037092",
        "rna_uuid": "35e2ac4f-1f27-4c48-9a15-9bb66a81b6ac",
        "label": "Good",
    },
    "TCGA-US-A77G": {
        "svs_uuid": "93b92629-c374-451d-bc92-f0ea78e70e2b",
        "rna_uuid": "c5b35b34-cf5e-4168-8aec-79a7df72da0e",
        "label": "Poor",
    },
}

for sample_id, info in SAMPLES.items():
    print(f"{sample_id} ({info['label']} candidate):")
    download_if_needed(
        f"{GDC_BASE}/{info['svs_uuid']}",
        DEMO_WSI_DIR / f"{sample_id}.svs",
    )
    download_if_needed(
        f"{GDC_BASE}/{info['rna_uuid']}",
        DEMO_OMICS_DIR / f"{sample_id}.tsv",
    )

print(f"\nWSI directory:   {DEMO_WSI_DIR}")
print(f"Omics directory: {DEMO_OMICS_DIR}")
print(f"Total slides:    {len(list(DEMO_WSI_DIR.glob('*.svs')))}")


In [ ]:
# Inspect the downloaded slides and RNA-seq files with the SDK helpers.
# display_slide_info prints dimensions / MPP / objective via the SDK WSI reader
# and returns a thumbnail; display_rna_summary reports the RNA-seq file stats.
for sample_id in SAMPLES:
    print(f"── {sample_id} ──")
    display_slide_info(DEMO_WSI_DIR / f"{sample_id}.svs")
    display_rna_summary(DEMO_OMICS_DIR / f"{sample_id}.tsv")
    print()


## 2. Configuration

In [ ]:
# Point this at your deployed M-Optimus server. Defaults to a local on-premise
# container; override with the API_URL environment variable (e.g. a SageMaker
# proxy or a remote host) without editing the notebook.
API_URL = os.environ.get("API_URL", "http://localhost:8080")
# Inference outputs are written under the same gitignored .data/ root as the
# downloaded slides (see Section 1).
OUTPUT_DIR = DATA_ROOT / "outputs"
EXPERIMENT = "tcga_paad_msln"
VARIANT = "mini"
RUN = 1
MASK_THRESHOLD = 0.5

# Genes of interest.
MSLN_GENE = "MSLN"
STROMAL_GENES = ["COL1A1", "POSTN", "ACTA2", "VCAN", "LAMC2"]
ALL_GENES = [MSLN_GENE] + STROMAL_GENES

# Stromal barrier parameters.
MSLN_PERCENTILE = 75
STROMAL_PERCENTILE = 85
BARRIER_K = 10

print_config(
    **{
        "API URL": API_URL,
        "Output": OUTPUT_DIR,
        "Genes": ALL_GENES,
        "Barrier K": BARRIER_K,
    }
)

# Confirm the inference server is reachable and report its loaded models.
check_server(API_URL)


## 3. Build the Cohort

Discovers the WSIs and builds a cohort. We deliberately run image-only
prediction first (Section 4), then link bulk RNA and re-predict, so we can
compare H&E-only vs. multimodal (H&E + bulk RNA) outputs side by side —
the exact refinement M-Optimus is designed for.


In [ ]:
cohort = Cohort.from_directories(wsi_dir=DEMO_WSI_DIR)

print(cohort.summary())
print(f"\nWSI IDs: {cohort.wsi_ids}")
print(f"Modalities: {cohort[0].available_modalities}")


## 4. Tissue Masking

Creates the `Inference` object for M-Optimus and generates a **tissue mask** per
slide, separating tissue from background so the expensive prediction step only
runs on tissue tiles. Tissue masking is **required** for prediction either way
(with or without bulk RNA). Masks are cached to the workspace and reused.


In [ ]:
infer = Inference(
    model_name=Models.M_OPTIMUS,
    cohort=cohort,
    variant=VARIANT,
    api_url=API_URL,
    tissue=True,
    mask_threshold=MASK_THRESHOLD,
    output_path=OUTPUT_DIR,
    experiment=EXPERIMENT,
    run=RUN,
    timeout=120.0,
    workers=1,
)

print(f"Workspace: {infer.workspace}")
print(f"WSIs:      {len(cohort)}")

In [ ]:
# Generate tissue masks for all slides.
infer.tissue()
print("Tissue masks generated.")

In [ ]:
# Visualize each slide beside its tissue mask (the mask filters out background
# tiles before the expensive prediction step). infer.tissue() cached each mask
# and recorded its path on the cohort record; plot_slide_and_mask renders them.
for record in cohort:
    if record.mask_path is not None:
        plot_slide_and_mask(
            record.wsi_path,
            record.mask_path,
            threshold=MASK_THRESHOLD,
        )


## 4a. Image-only Prediction *(optional)*

Runs M-Optimus prediction from **H&E alone** (no bulk RNA). This step is
**optional** — it exists only to unlock the H&E-only vs. multimodal comparison
in Section 6b, and it roughly doubles total inference time. Skip this cell (and
Section 6b) if you only need the bulk-RNA-informed results; the analysis in
Sections 5–9 runs on the multimodal output produced in Section 4b. Modality-aware
tracking stores this as a separate stage, so the image-only and multimodal
outputs remain available side by side.


In [ ]:
# OPTIONAL — image-only baseline: predict from H&E alone (no bulk RNA linked yet).
# This exists solely to enable the H&E-only vs. multimodal comparison in
# Section 6b. Skip this cell (and Section 6b) if you only need the bulk-RNA
# results — it roughly doubles inference time.
infer.run(mode="predict")
infer.report()


## 4b. Add Bulk RNA and Re-predict (Multimodal)

Links the matched bulk RNA-seq files (matched by filename stem, e.g.
`TCGA-F2-A8YN.svs` ↔ `TCGA-F2-A8YN.tsv`) and re-runs prediction with
`force=True`. Modality-aware tracking keeps the image-only output separate, so
both stages remain available for comparison. Bulk RNA is sent as
TPM-normalized values — the server applies `log1p` on input and `expm1` on
output, so no manual transform is needed here.


In [ ]:
# Link matched bulk RNA (GDC/TCGA gene-quantification TSVs: use tpm_unstranded
# and strip Ensembl versions), then re-predict with bulk context.
linked = cohort.link_bulk_rna(
    DEMO_OMICS_DIR,
    gene_column="gene_id",
    value_column="tpm_unstranded",
    strip_version=True,
)
print(f"Linked {linked} / {len(cohort)} records with bulk RNA.")
print(f"Modalities: {cohort[0].available_modalities}")

# force=True re-runs prediction; the image-only stage is preserved separately.
infer.run(mode="predict", force=True)
infer.report()

# Persist the pipeline config to <workspace>/config.yaml so the exact run can be
# reconstructed later with Inference.from_workspace(infer.workspace).
infer.save_config()


## 5. Load Predictions and Build Gene Index

Reads prediction zarr outputs for both slides and maps gene symbols
to prediction array column indices using the `output_gene_names` array
stored in each zarr.

In [ ]:
# Load predictions for all slides with the SDK's built-in Zarr loader.
# The multimodal (image + bulk_rna) output drives the analysis. The image-only
# output is optional (Section 4) and, when present, kept for the Section 6b
# comparison.
slide_data = {}
HAS_IMAGE_ONLY = True  # Set False automatically if the optional stage is missing.


def _coords_to_level0(coords: np.ndarray, meta: dict) -> np.ndarray:
    """Rescales tile coords from extraction-MPP space to level-0 slide pixels.

    The pipeline stores tile coordinates at the extraction resolution
    (``slide_dimensions_at_mpp``), but the SDK overlay helpers position tiles
    against the full-resolution slide thumbnail (``slide_dimensions``). Without
    this rescale the heatmap is squashed into the top-left of the thumbnail.
    """
    dims = meta.get("slide_dimensions")
    dims_at_mpp = meta.get("slide_dimensions_at_mpp")
    if not dims or not dims_at_mpp:
        return coords
    scale = np.array(dims, dtype=float) / np.array(dims_at_mpp, dtype=float)
    return coords * scale


for record in cohort:
    wsi_id = record.wsi_id
    output = record.outputs.get(infer.model_name)

    multimodal_path = output.get_stage_path("predict", ["image", "bulk_rna"])

    # load_zarr_output returns outputs, coords, gene_names (decoded from
    # output_gene_names), plus thumbnail/tissue_mask/metadata when present.
    loaded = load_zarr_output(multimodal_path)

    # SDK outputs are in expm1(log1p) count space; convert back to log1p for analysis.
    preds = np.log1p(loaded["outputs"])
    coords = loaded["coords"]
    gene_names = loaded["gene_names"]

    slide_data[wsi_id] = {
        "preds": preds,
        # Extraction-MPP coords for the (scale-invariant) spatial analysis.
        "coords": coords,
        # Level-0 coords for the thumbnail overlays (SDK plot helpers).
        "coords_plot": _coords_to_level0(coords, loaded["metadata"]),
        "gene_names": gene_names,
        "label": SAMPLES[wsi_id]["label"],
    }

    # Optional image-only stage (Section 4) — only load it if it was produced.
    image_only_path = output.get_stage_path("predict", ["image"])
    if image_only_path is not None and Path(image_only_path).exists():
        image_only = load_zarr_output(image_only_path)
        slide_data[wsi_id]["preds_image_only"] = np.log1p(image_only["outputs"])
        slide_data[wsi_id]["coords_image_only"] = _coords_to_level0(
            image_only["coords"], image_only["metadata"]
        )
    else:
        HAS_IMAGE_ONLY = False

    print(f"{wsi_id}: {preds.shape[0]:,} tiles × {preds.shape[1]:,} genes")

if not HAS_IMAGE_ONLY:
    print("\nNote: image-only stage not found — skip Section 6b (optional comparison).")

# Map gene symbols to Ensembl IDs with the SDK helper (strips version suffixes),
# then to prediction-array column indices via the decoded output gene names.
first_sample = next(iter(SAMPLES))
symbol_to_ensembl = read_gene_mapping(
    DEMO_OMICS_DIR / f"{first_sample}.tsv",
    symbol_column="gene_name",
    id_column="gene_id",
    strip_version=True,
)
ref_genes = slide_data[next(iter(slide_data))]["gene_names"]
ensembl_to_idx = {g: i for i, g in enumerate(ref_genes)}

# Resolve gene indices for MSLN + stromal genes. Fail fast with the full list of
# missing genes rather than proceeding to a later KeyError on gene_indices[...].
gene_indices = {}
missing_genes = []
for gene_symbol in ALL_GENES:
    ensembl_id = symbol_to_ensembl.get(gene_symbol)
    if ensembl_id and ensembl_id in ensembl_to_idx:
        gene_indices[gene_symbol] = ensembl_to_idx[ensembl_id]
    else:
        missing_genes.append(gene_symbol)

if missing_genes:
    raise KeyError(
        f"Genes not found in the model output set: {missing_genes}. "
        "Check the symbols in ALL_GENES against the model's output_gene_names."
    )

# Cache per-tile MSLN expression so downstream cells can run in any order.
for data in slide_data.values():
    data["msln_expr"] = data["preds"][:, gene_indices[MSLN_GENE]]

print(f"\nGene indices resolved: {list(gene_indices.keys())}")
for g, idx in gene_indices.items():
    print(f"  {g}: column {idx}")


## 6. Spatial MSLN Expression — Same Bulk, Different Spatial

Both patients are MSLN-high by bulk RNA-seq. But spatial gene expression
reveals dramatically different patterns. Each plot is annotated with the
bulk expression value and spatial heterogeneity metrics (CV, Moran's I)
to show that bulk averaging hides critical information.

In [ ]:
# Bulk MSLN expression per slide, read directly from each downloaded RNA-seq
# TSV (self-contained — no external cohort matrix needed). Values are
# TPM-normalized; we report log2(TPM+1) to match how bulk/IHC studies quote them.
bulk_msln = {}
for wsi_id in slide_data:
    rna = pd.read_csv(DEMO_OMICS_DIR / f"{wsi_id}.tsv", sep="\t", comment="#")
    row = rna.loc[rna["gene_name"] == MSLN_GENE, "tpm_unstranded"]
    bulk_msln[wsi_id] = float(np.log2(row.iloc[0] + 1)) if len(row) else np.nan

# Compute spatial heterogeneity metrics.
for wsi_id, data in slide_data.items():
    msln = data["msln_expr"]
    data["msln_cv"] = float(msln.std() / (msln.mean() + 1e-8))
    coords = data["coords"]
    if len(coords) > BARRIER_K + 1:
        tree = cKDTree(coords)
        _, nn_idx = tree.query(coords, k=BARRIER_K + 1)
        nn_idx = nn_idx[:, 1:]
        x = msln - msln.mean()
        lag = x[nn_idx].mean(axis=1)
        data["msln_morans_i"] = float(np.sum(x * lag) / (np.sum(x**2) + 1e-8))
    else:
        data["msln_morans_i"] = np.nan

# Side-by-side MSLN spatial expression — both slides in one figure, each
# overlaid on its own thumbnail with the SDK's spatially-accurate helper.
# Titles carry the bulk value and spatial-heterogeneity scores, so the shared
# "MSLN-high by bulk" story can be read against the differing spatial patterns.
msln_idx = gene_indices[MSLN_GENE]
fig, axes = plt.subplots(1, len(slide_data), figsize=(8 * len(slide_data), 7))

for ax, (wsi_id, data) in zip(axes, slide_data.items()):
    plot_gene_overlay(
        data["preds"][:, msln_idx],
        data["coords_plot"],
        DEMO_WSI_DIR / f"{wsi_id}.svs",
        title=(
            f"{wsi_id} ({data['label']} ADC candidate)\n"
            f"Bulk MSLN = {bulk_msln[wsi_id]:.2f} log2(TPM+1)\n"
            f"Spatial CV = {data['msln_cv']:.3f} | Moran's I = {data['msln_morans_i']:.3f}"
        ),
        cmap="inferno",
        ax=ax,
    )

fig.suptitle(
    "Same bulk MSLN expression — very different spatial patterns", fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()


## 6b. What Bulk RNA Adds — Multimodal vs. H&E-only

M-Optimus predicts spatial expression from H&E alone, and refines it when bulk
RNA is supplied. Here we compare the two MSLN predictions per slide — image-only
vs. image + bulk RNA — to show how the bulk readout sharpens the spatial signal
without any retraining.


In [ ]:
# MSLN predicted with vs. without bulk RNA, on each slide's own thumbnail.
# Requires the optional image-only stage from Section 4.
if not HAS_IMAGE_ONLY:
    print("Skipped: image-only prediction (Section 4) was not run.")
else:
    msln_idx = gene_indices[MSLN_GENE]
    for wsi_id, data in slide_data.items():
        plot_gene_overlay_comparison(
            data["preds"],
            data["coords_plot"],
            data["preds_image_only"],
            data["coords_image_only"],
            gene_idx=msln_idx,
            wsi_path=DEMO_WSI_DIR / f"{wsi_id}.svs",
            gene_name="MSLN",
            label_a="With bulk RNA (multimodal)",
            label_b="Without bulk RNA (image only)",
            slide_id=f"{wsi_id} ({data['label']})",
        )


## 7. Compute Desmoplastic Score and Stromal Barrier

For each slide:
- **Desmoplastic score**: z-score each stromal gene within the slide, then
  average across genes → per-tile stroma intensity.
- **Stromal barrier**: among MSLN-high tiles (≥75th percentile), what fraction
  have K=10 nearest neighbors with high desmoplastic score (≥85th percentile)?

Both metrics are computed entirely within each slide (no cross-slide dependence).

In [ ]:
results = []

for wsi_id, data in slide_data.items():
    preds = data["preds"]
    coords = data["coords"]
    n_tiles = len(preds)

    # Extract MSLN expression.
    msln_expr = preds[:, gene_indices[MSLN_GENE]]

    # Extract stromal gene columns and compute desmoplastic score.
    stromal_preds = np.column_stack([preds[:, gene_indices[g]] for g in STROMAL_GENES])
    stromal_z = np.zeros_like(stromal_preds)
    for col in range(stromal_preds.shape[1]):
        s = stromal_preds[:, col]
        std = s.std()
        stromal_z[:, col] = (s - s.mean()) / (std if std > 1e-8 else 1.0)
    desmo_score = stromal_z.mean(axis=1)

    # Compute stromal barrier metric.
    stromal_barrier = np.nan
    if n_tiles >= BARRIER_K + 1:
        msln_thresh = np.percentile(msln_expr, MSLN_PERCENTILE)
        desmo_thresh = np.percentile(desmo_score, STROMAL_PERCENTILE)

        msln_high_mask = msln_expr >= msln_thresh
        n_msln_high = msln_high_mask.sum()

        if n_msln_high > 0:
            tree = cKDTree(coords)
            msln_high_coords = coords[msln_high_mask]
            _, nn_idx = tree.query(msln_high_coords, k=BARRIER_K + 1)
            nn_idx = nn_idx[:, 1:]  # Exclude self.
            nn_desmo = desmo_score[nn_idx].mean(axis=1)
            stromal_barrier = float((nn_desmo >= desmo_thresh).sum() / n_msln_high)

    # Store results.
    data["msln_expr"] = msln_expr
    data["desmo_score"] = desmo_score
    data["stromal_barrier"] = stromal_barrier

    results.append(
        {
            "patient": wsi_id,
            "label": data["label"],
            "n_tiles": n_tiles,
            "mean_msln": float(msln_expr.mean()),
            "cv_msln": float(msln_expr.std() / (msln_expr.mean() + 1e-8)),
            "mean_desmo": float(desmo_score.mean()),
            "stromal_barrier": stromal_barrier,
        }
    )

results_df = pd.DataFrame(results)

## 8. The Stromal Barrier — Why MSLN Location Alone Is Not Enough

Knowing that a patient is MSLN-high *and* seeing the spatial MSLN pattern
only gets you so far. Three observations:

1. **MSLN spatial location alone is insufficient** — both slides show
   spatially concentrated MSLN-high regions, but the clinical outcome
   depends on what *surrounds* those regions.
2. **Desmoplastic score can be similar between patients** — mean stroma
   intensity may not differ much slide-to-slide.

3. **Co-localization is the key** — the stromal barrier metric asks:
   among MSLN-high tiles, are their immediate neighbors stromal-dense?
   This captures whether the ADC can physically reach its target.

In [ ]:
# Per slide, overlay MSLN expression and the derived desmoplastic score on the
# slide thumbnail. MSLN is a model output gene; the desmoplastic score is a
# per-tile derived metric, so both are drawn with the SDK's overlay helper
# (spatially accurate on the real thumbnail).
for wsi_id, data in slide_data.items():
    wsi_path = DEMO_WSI_DIR / f"{wsi_id}.svs"
    barrier = data["stromal_barrier"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plot_gene_overlay(
        data["msln_expr"],
        data["coords_plot"],
        wsi_path,
        title=f"{wsi_id} ({data['label']})\nMSLN expression",
        cmap="inferno",
        ax=axes[0],
    )
    plot_gene_overlay(
        data["desmo_score"],
        data["coords_plot"],
        wsi_path,
        title=f"{wsi_id} ({data['label']})\nDesmoplastic score | Barrier = {barrier:.3f}",
        cmap="YlOrRd",
        vmin=None,
        ax=axes[1],
    )
    fig.suptitle(
        "MSLN expression vs desmoplastic stroma — "
        "the stromal barrier metric captures co-localization, not just intensity",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()


## 9. Zoomed View — Tiles Driving the MSLN Signal

To make the co-localization concept concrete, we pull the individual tiles with
the highest and lowest predicted MSLN expression for each patient using the
SDK's `plot_top_gene_tiles` helper. This reads the actual H&E tiles back from
the slide, so you can inspect the morphology behind the prediction.


In [ ]:
# Highest- and lowest-MSLN tiles per slide, read back from the H&E.
# NOTE: plot_top_gene_tiles reads tile regions via read_region(resolution=0.5 MPP),
# which interprets the location at the *extraction* resolution — so it takes the
# native extraction-space coords, not the level-0 coords the overlay helpers use.
msln_idx = gene_indices[MSLN_GENE]
for wsi_id, data in slide_data.items():
    print(f"{wsi_id} ({data['label']} ADC candidate):")
    plot_top_gene_tiles(
        data["preds"],
        data["coords"],
        gene_idx=msln_idx,
        wsi_path=DEMO_WSI_DIR / f"{wsi_id}.svs",
        gene_name="MSLN",
        n_top=6,
        n_bottom=6,
    )


## Summary

This notebook demonstrated that **H&E-predicted spatial gene expression**
(via M-Optimus + bulk RNA) can potentially help stratify MSLN-high pancreatic cancer
patients for ADC therapy:

| Patient | Stromal Barrier | ADC Suitability |
|---------|----------------|----------------|
| TCGA-F2-A8YN | Low | ✅ Good candidate |
| TCGA-US-A77G | High | ❌ Poor candidate |

By being able to visualize both the spatial MSLN expression patterns, as well as the
surrounding stromal environment, we can refine patient selection and / or identify
patients who would require additional combination therapies (e.g. stroma-disrupting
PEGPH20 hyaluronidase).

## Extensions

Importantly, this type of analysis can expand to other modes of ADC-therapeutic failure,
such as TAM/Fc-receptor sinks where macrophages soak up the drug before it reaches
the tumor cell and efflux induction, where the cell pumps the payload back out before it
can kill it.
